# glcuda M2 — T4 hardware validation

First real-hardware run of the GwenLand **glcuda** CUDA backend (`ArchGLML_X2` M2).
Everything below the FFI/kernel layer has only ever been tested on a machine with **no GPU** —
the PTX has never been JIT-compiled. This notebook is the gate.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. The T4 is `sm_75`
(Turing), which clears glcuda's `sm_70+` floor.

Run the cells top to bottom. Each numbered section prints a clear banner; copy the
output back and I'll interpret it. If a cell fails, **stop and report it** — later
cells depend on earlier ones.

## 0 · Confirm the GPU

If this says anything other than a Tesla T4 (or another `sm_70+` card), fix the runtime type first.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv
print()
!nvidia-smi | head -n 12

## 1 · Install a Rust toolchain

Colab has no Rust by default. This installs a recent stable `rustc`/`cargo` via rustup
(~30–60 s). glcuda needs **no** CUDA toolkit or `nvcc` — it loads the driver
(`libcuda.so.1`, already present on the T4 image) at runtime and ships hand-written PTX.

In [ ]:
import os
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!rustc --version && cargo --version

## 2 · Clone the repo (M2 branch)

The glcuda M2 work lives on the `feature/m2-glcuda` branch — **not** `main`.
Public repo, so no credentials needed.

In [ ]:
%cd /content
!rm -rf gwenland-ai
!git clone --depth 1 https://github.com/JinXSuperSolo/gwenland-ai.git
%cd /content/gwenland-ai
!git log --oneline -1
!echo '--- glcuda source present? ---'
!ls -1 glcuda/src glcuda/src/kernels glcuda/tests

## 3 · Sanity: confirm the driver is loadable from Rust's view

Quick check that the CUDA driver library the engine `dlopen`s at runtime is where it
expects. (Informational — the real proof is the tests below.)

In [ ]:
!ldconfig -p | grep -i libcuda || echo 'libcuda not in ldconfig cache'
!ls -l /usr/lib/x86_64-linux-gnu/libcuda.so* 2>/dev/null || true

## 4 · Build glcuda (release)

First build compiles the workspace deps too — expect **3–6 minutes**. Release mode so the
tok/s numbers later are meaningful. A clean finish here means the Rust ↔ driver FFI links.

In [ ]:
!cargo build --release -p glcuda 2>&1 | tail -n 20

## 5 · Host-only tests (no GPU path)

The dequant/bump-allocator/sampler unit tests. These already pass on a GPU-less machine;
running them here confirms the toolchain is sane before we touch the device.

In [ ]:
!cargo test --release -p glcuda --lib 2>&1 | tail -n 40

## 6 · ⭐ GPU kernel parity suite

**This is the first real test of the PTX on hardware.** Each of the seven SIMT kernels is
run on the T4 and compared to the glproc CPU ground truth within the per-operation ε from
`ArchGLML_X2` §8 (matmul 1e-5, rmsnorm 1e-6, softmax 1e-5, rope 1e-7, swiglu 1e-6). Also
covers the Q8_0 quantized GEMV and the VRAM-leak check (backend buffer returns memory
exactly).

`--test-threads=1` is **required**: the VRAM leak test reads free memory before/after and
must not run concurrently with other allocating tests.

If a kernel had a PTX bug, this is where it surfaces (either a JIT error like
`cuModuleLoadData(PTX JIT) failed` or a parity assertion `gpu X vs cpu Y`).

In [ ]:
!cargo test --release -p glcuda --test parity -- --test-threads=1 --nocapture 2>&1 | tail -n 60

## 7 · ⭐ End-to-end forward-pass parity

The whole transformer, both engines, identical tiny f32 model (GQA + NeoX RoPE + qwen2
biases + qwen3 head-norms + SwiGLU): glcuda's logits must match glproc within 1e-3 **and**
pick the same argmax, and a greedy generation must produce **token-identical** output.
This proves the static-graph runner, KV cache, and upload path are correct end to end.

In [ ]:
!cargo test --release -p glcuda --test forward -- --test-threads=1 --nocapture 2>&1 | tail -n 40

## 8 · (Optional) Real model — coherence + tok/s

Everything above uses synthetic weights. This section runs a **real Qwen2.5-0.5B GGUF**
through the engine to check the output is coherent text and to get a first tok/s number on
the T4.

**Use a Q8_0 quant** — glcuda keeps Q8_0 tensors quantized in VRAM (native kernel) and
dequantizes any k-quant to dense f32, so Q8_0 is both the fast and the recommended M2
format. The download is ~500 MB.

In [ ]:
# Grab a Q8_0 Qwen2.5-0.5B-Instruct GGUF from Hugging Face (no auth needed).
%cd /content/gwenland-ai
!wget -q --show-progress -O qwen2.5-0.5b-q8.gguf \
  https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q8_0.gguf
!ls -lh qwen2.5-0.5b-q8.gguf

### 8a · Minimal driver program

There's no glcuda-specific CLI yet (wiring it into `glcli`'s fallback chain is the next M2
task), so we write a tiny example that drives the engine directly: init → load → stream,
printing tokens and the prefill/decode tok/s the engine reports.

In [ ]:
example = r'''
use std::time::Instant;
use glcore::engine_trait::{GlEngine, InferInput};
use glcuda::GlcudaEngine;

fn main() -> Result<(), Box<dyn std::error::Error>> {
    let path = std::env::args().nth(1).expect("usage: run_glcuda <model.gguf> [prompt]");
    let prompt = std::env::args().nth(2)
        .unwrap_or_else(|| "Explain what a GPU is in one sentence.".to_string());

    let mut engine = GlcudaEngine::new();
    let t = Instant::now();
    engine.init()?;
    println!("[init] {:.2}s", t.elapsed().as_secs_f64());

    let t = Instant::now();
    engine.load_model(&path)?;
    println!("[load] {:.2}s", t.elapsed().as_secs_f64());

    // encode_chat applies the model's ChatML template internally.
    let ids = engine.encode_chat(&prompt)?;
    println!("[prompt] {} tokens", ids.len());

    let input = InferInput {
        token_ids: ids,
        max_new_tokens: 128,
        temperature: 0.7,
        top_k: 40,
        top_p: 0.95,
        repeat_penalty: 1.1,
    };

    print!("\n=== output ===\n");
    let out = engine.stream(input, &|_id, piece| { print!("{piece}"); use std::io::Write; std::io::stdout().flush().ok(); })?;
    println!("\n=== stats ===");
    println!("prompt tokens : {}", out.prompt_tokens);
    println!("gen tokens    : {}", out.tokens_generated);
    println!("prefill       : {:.1} ms ({:.1} tok/s)",
        out.prefill_ms, out.prompt_tokens as f64 / (out.prefill_ms/1e3).max(1e-9));
    println!("decode        : {:.1} ms ({:.1} tok/s)",
        out.generation_ms, out.tokens_generated as f64 / (out.generation_ms/1e3).max(1e-9));
    Ok(())
}
'''
import os
os.makedirs('/content/gwenland-ai/glcuda/examples', exist_ok=True)
with open('/content/gwenland-ai/glcuda/examples/run_glcuda.rs', 'w') as f:
    f.write(example)
print('wrote glcuda/examples/run_glcuda.rs')

### 8b · Add glcore as a dependency for the example

The example needs `glcore` (for `GlEngine`/`InferInput`). glcuda already depends on it
internally, but examples need it declared as a dev-dependency. This patches the
`Cargo.toml` idempotently.

In [ ]:
import re
p = '/content/gwenland-ai/glcuda/Cargo.toml'
s = open(p).read()
if 'glcore' not in s.split('[dev-dependencies]')[-1]:
    if '[dev-dependencies]' in s:
        s = s.replace('[dev-dependencies]', '[dev-dependencies]\nglcore = { path = "../glcore" }', 1)
    else:
        s += '\n[dev-dependencies]\nglcore = { path = "../glcore" }\n'
    open(p, 'w').write(s)
print(open(p).read())

### 8c · Run the real model

Watch for: (1) coherent English, not garbage — proves numerical correctness on a real
model; (2) the decode tok/s. **The M2 goal is correctness, not speed** — the decode loop
issues ~1000 un-fused kernel launches per token (attention is one launch per head), so
expect a modest number. That's the M2.1 optimization target, not a bug.

In [ ]:
!cargo run --release -p glcuda --example run_glcuda -- qwen2.5-0.5b-q8.gguf \
  "Explain what a GPU is in one sentence." 2>&1 | tail -n 60

## What to report back

Paste the output of:
- **§0** — the exact GPU / driver / compute-cap
- **§6** — full parity-suite result (this is the headline: did the PTX JIT and match ε?)
- **§7** — end-to-end forward parity result
- **§8c** — if you ran it: the generated text + the prefill/decode tok/s

Any failure — a JIT error, a parity assertion, or a compile error in §8 — is exactly the
signal I need. Copy it verbatim.

## 9 - Microbenchmark: where does decode time actually go?

Attention fusion did not move decode tok/s, so per-launch overhead was NOT the
bottleneck. This bench measures the real suspects at Qwen2.5-0.5B decode sizes:
achievable memory bandwidth, GEMV throughput (batched vs per-call-synced - the gap
is serialization/latency), and the cost of the 96 per-token KV-cache DtoD copies.

**Report the whole output** - it tells us which kernel/pattern to fix next.

In [ ]:
!cargo run --release -p glcuda --example bench 2>&1 | tail -n 25

## 10 - nsys: profile decode and settle (A) vs (B)

Five pipeline/kernel theories have measured flat (attention fusion, CUDA graphs,
gate+up fusion, dp4a Q8_0). Data says: pipeline is NOT the bottleneck, FFN dominates
the token. Two competing explanations remain:

- **(A)** the FFN GEMV kernel is efficient in isolation but degrades in the decode loop
  (occupancy, cache, serialization) -> fix the kernel.
- **(B)** the GEMV is already at the memory-bandwidth ceiling -> the only lever is
  moving fewer bytes (FFN weights in Q4_K instead of Q8_0).

Decisive test, NO perf-counters needed: compare each GEMV's **in-loop node duration**
(from the captured decode graph, expanded via `--cuda-graph-trace=node`) against the
**isolated** duration `bench.rs` already reports. Same -> (B). Slower in-loop -> (A).

### 10a - Install Nsight Systems CLI

The driver-only T4 image has no `nsys`. Pull just the CLI from NVIDIA's apt repo
(already configured on the Colab image).

In [ ]:
# nsys is not on the driver image. FIRST check whether Colab's CUDA toolkit already
# ships it (common), otherwise register NVIDIA's apt repo via cuda-keyring and install
# just the CLI. Either way, put the binary on PATH.
import glob, os, shutil
cand = shutil.which('nsys')
if not cand:
    hits = (glob.glob('/usr/local/cuda*/bin/nsys')
            + glob.glob('/opt/nvidia/nsight-systems*/*/bin/nsys')
            + glob.glob('/opt/nvidia/nsight-systems*/bin/nsys'))
    if hits:
        cand = hits[0]
if not cand:
    !wget -qO /tmp/cuda-keyring.deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
    !dpkg -i /tmp/cuda-keyring.deb
    # apt-get update may warn about the unrelated r2u repo; only NVIDIA's list matters.
    !apt-get update -o Dir::Etc::sourcelist="sources.list.d/cuda-ubuntu2204-x86_64.list" -o Dir::Etc::sourceparts="-" -o APT::Get::List-Cleanup="0"
    !apt-get install -y nsight-systems-cli
    hits = glob.glob('/opt/nvidia/nsight-systems*/*/bin/nsys') + glob.glob('/opt/nvidia/nsight-systems*/bin/nsys')
    cand = hits[0] if hits else shutil.which('nsys')
if cand:
    os.environ['PATH'] = os.path.dirname(cand) + ':' + os.environ['PATH']
print('nsys ->', cand or 'NOT FOUND')
!nsys --version || echo "nsys unavailable - use Section 11 (host/GPU split) instead, it needs no nsys"

### 10b - Build the example binary

Profile the compiled binary directly, not `cargo run`, so rustc/cargo don't pollute
the timeline.

In [ ]:
%cd /content/gwenland-ai
!cargo build --release -p glcuda --example run_glcuda 2>&1 | tail -n 5

### 10c - Profile a decode run, expand the CUDA graph into per-kernel nodes

Decode replays a captured graph, so without `--cuda-graph-trace=node` nsys shows one
opaque node. A longer generation gives many steady-state decode replays to average.

In [ ]:
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!rm -f /content/glcuda_decode.nsys-rep /content/glcuda_decode.sqlite
!nsys profile \
  --trace=cuda \
  --cuda-graph-trace=node \
  --force-overwrite=true \
  -o /content/glcuda_decode \
  target/release/examples/run_glcuda qwen2.5-0.5b-q8.gguf \
  "Write one detailed paragraph explaining why the ocean appears blue." 2>&1 | tail -n 15

### 10d - Per-kernel breakdown (in-loop durations)

`cuda_gpu_kern_sum` = total GPU time, instances, and **avg per-invocation** by kernel.
This tells us (1) which kernel class dominates decode and (2) the average in-loop
duration of `gl_gemv_q8_0` to compare against bench's isolated numbers.

Note: `gl_gemv_q8_0` is used for qkv, gate_up, down AND lm_head, so its rows share a
name; the min/max/avg spread reflects the different matmul sizes (lm_head is the
largest single GEMV).

In [ ]:
!nsys stats --report cuda_gpu_kern_sum /content/glcuda_decode.nsys-rep 2>&1 | tail -n 40

### 10e - Per-invocation node trace (separate FFN from lm_head/attn by size)

Groups every kernel launch by name + duration so the three FFN GEMVs per layer
(gate_up, down) are distinguishable from qkv and the one big lm_head GEMV.

In [ ]:
!nsys stats --report cuda_gpu_trace /content/glcuda_decode.nsys-rep 2>&1 | grep -E "gemv_q8_0|attn_decode|silu_mul|rms_norm|Duration|Name" | sort | uniq -c | sort -rn | head -40

### 10f - (Optional) DRAM throughput via GPU metrics

Direct confirmation of BW-bound: if DRAM throughput sits ~85-95% during decode, that's
(B). **This needs GPU performance counters, which Colab often locks** -- if it errors
with a permission/counter message, skip it; 10d+10e already answer A vs B.

In [ ]:
!rm -f /content/glcuda_decode_gm.nsys-rep /content/glcuda_decode_gm.sqlite
!nsys profile \
  --trace=cuda \
  --cuda-graph-trace=node \
  --gpu-metrics-devices=all \
  --force-overwrite=true \
  -o /content/glcuda_decode_gm \
  target/release/examples/run_glcuda qwen2.5-0.5b-q8.gguf \
  "Write one detailed paragraph explaining why the ocean appears blue." 2>&1 | tail -n 15

### 10g - Re-run bench for the isolated baseline, then compare

Compare `gl_gemv_q8_0` avg from 10d against bench's isolated batched-GEMV time for the
same shapes. Interpretation:

- **in-loop ~= isolated** (both ~85-90% of ~266 GB/s achievable) -> **(B) BW-bound**.
  FFN dominates purely because it carries the most weight bytes. Only lever: fewer
  bytes -> prototype FFN weights in Q4_K (down/gate/up are the fattest tensors).
- **in-loop >> isolated** -> **(A)**. The loop degrades the kernel; look at occupancy
  (32-thread blocks starve the SM) -> multi-warp-per-row GEMV, not requantization.

Also sanity-check vs llama.cpp's 228 tok/s on this same T4/model: if our GEMVs already
match its per-matvec BW, the remaining gap is precision/bytes, not scheduling.

In [ ]:
!cargo run --release -p glcuda --example bench 2>&1 | tail -n 25

## 11 - Host vs GPU split (NO nsys needed) - the decisive measurement

The bench shows the isolated Q8_0 GEMVs sum to only ~2.7 ms/token, yet decode is
~6.4 ms/token (157 tok/s). So >50% of the token is NOT the matmuls. This run
attributes each decode token's wall time to **GPU** (waiting on the in-flight decode
graph) vs **HOST** (logits DtoH + repetition penalty + CPU sample over the 151,936
vocab, during which the GPU is idle).

`GLCUDA_PROFILE_DECODE=1` turns on the split (added to `Runner::generate`). A large
host share means the bottleneck is CPU sampling / serialization, NOT any GPU kernel -
which would explain why fusion, graphs and dp4a all measured flat.

In [ ]:
%cd /content/gwenland-ai
!git pull --ff-only 2>&1 | tail -2
!cargo build --release -p glcuda --example run_glcuda 2>&1 | tail -2
import os
os.environ['GLCUDA_PROFILE_DECODE'] = '1'
!GLCUDA_PROFILE_DECODE=1 target/release/examples/run_glcuda qwen2.5-0.5b-q8.gguf   "Write one detailed paragraph explaining why the ocean appears blue." 2>&1 | tail -n 20